# NPTEL Course: Grid Connected Power Converters — Operating Principles

## Simulation Study 2 — Single-Phase Converter with Switching Pole, DC Side Driven by a Current Source

**Author:** V. Seshadri Sravan Kumar, IIT Hyderabad

**Objective**: This notebook simulates the same single-phase switching-pole converter as Simulation Study 1, but with the DC side modeled as a **current source** rather than an ideal DC voltage source (battery). The output filter remains a purely **L filter** (inductor $L$ with parasitic resistance $R$).

In a practical renewable-energy-source (RES) interface, the current injected into the DC link contains a DC component together with harmonic components. In this simulation, that entire upstream behavior is represented by a prescribed current source, and source-side converter harmonics are neglected.

On the DC side, the link still consists of **two capacitors of equal value**, $c_1$ and $c_2$, connected in series, with no assumption that $v_{c1}(t)=v_{c2}(t)$. Unlike Simulation Study 1, the total DC link voltage $v_{dc}(t)=v_{c1}(t)+v_{c2}(t)$ is **no longer held constant**. Since a current source has high output impedance, it supplies the prescribed current and the resulting voltage adjusts accordingly, so $v_{dc}(t)$ evolves according to current balance at the switching pole.

Please refer to the modelling derivation carried out in the main lecture to see how the governing equations below were obtained from first principles.

### Governing Equations

**States:**

$$x(t) = \begin{bmatrix} i_f(t) \\ v_{c1}(t) \\ v_{c2}(t) \end{bmatrix}$$

**Switching function:** $q(t)\in\{+1,-1\}$, determined by the PWM modulator (reference vs. carrier comparison)

**Pole voltage:**

$$v_{ao}(t) = \frac{v_{c1}(t)-v_{c2}(t)}{2} + q(t)\cdot\frac{v_{c1}(t)+v_{c2}(t)}{2}$$

**AC side:**

$$\frac{di_f}{dt} = \frac{1}{L}\,v_{ao}(t) - \frac{R}{L}\,i_f(t) - \frac{1}{L}\,v_g(t)$$

**DC link:**

$$\frac{dv_{c1}}{dt} = \frac{1}{C_1}\left[i_s(t) - \frac{1+q(t)}{2}\,i_f(t)\right]$$

$$\frac{dv_{c2}}{dt} = \frac{1}{C_2}\left[\frac{1-q(t)}{2}\,i_f(t) + i_s(t)\right]$$

where $i_s(t)$ is a **prescribed source current**, not an algebraic output.

In [ ]:
# Importing required packages

import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

# Plotting style settings (applied globally to all subsequent plots)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid']      = True
plt.rcParams['grid.alpha']     = 0.3
plt.rcParams['font.size']      = 10

# Enforce full axis box (top/right spines and ticks) on all plots/subplots
plt.rcParams['axes.spines.top']   = True
plt.rcParams['axes.spines.right'] = True
plt.rcParams['xtick.top']         = True
plt.rcParams['ytick.right']       = True
plt.rcParams['xtick.direction']   = 'in'
plt.rcParams['ytick.direction']   = 'in'

### System Parameters

The values below are representative of a single-phase grid-connected inverter rated around 3 kW and connected to a 220 V, 50 Hz grid.

The modulation signal is defined as $m(t)=M\sin(\omega_1 t+\phi)$.

`Vdc_nominal` is not a fixed model state. It is used only to set initial capacitor voltages and to estimate the DC component of the source current $i_s(t)$. The actual DC-link voltage, $v_{dc}(t)=v_{c1}(t)+v_{c2}(t)$, evolves with time in this model.

In [ ]:
Vg_rms       = 220.0       # grid RMS voltage (V)
f1           = 50.0        # grid fundamental frequency (Hz)

Vdc_nominal  = 700.0       # nominal/reference DC link voltage (V) -- used only for
                           # initial conditions and for sizing the current source

M            = 0.89        # modulation index (adjustable)
phi          = -3*np.pi/4  # phase of modulating signal m(t), in radians (adjustable)

fsw          = 4000.0      # switching frequency (Hz)
m_f          = fsw/f1      # carrier ratio (frequency modulation ratio)

L            = 45e-3       # filter inductance (H)
R            = 0.15        # filter resistance (ohm)

C1           = 2200e-6      # DC link capacitor 1 (F)
C2           = 2200e-6      # DC link capacitor 2 (F)

### Time Discretization

The sampling interval is chosen from the required number of points per switching period. A sufficient number of samples per switching cycle helps capture ripple clearly and accurately.

In [ ]:
n_cycles          = 40                       # number of fundamental cycles to simulate
n_pts_per_Tsw     = 40                       # number of time instants per switching cycle

t_end             = n_cycles/f1              # total simulation time (s)
t_step            = (1.0/fsw)/n_pts_per_Tsw  # time step (s)

t = np.arange(0, t_end, t_step)              # time vector

### State Vector and Initial Conditions

The state vector is

$$x(t)=\begin{bmatrix} i_f(t) \\ v_{c1}(t) \\ v_{c2}(t) \end{bmatrix}.$$

### Near-Steady-State Initialization

As in Simulation Study 1, starting with $i_f(0)=0$ and $v_{c1}(0)=v_{c2}(0)=V_{dc,nominal}/2$ introduces a long $L/R$ transient.

To reduce startup transients, $i_f(0)$ is initialized from a fundamental-frequency phasor estimate. The corresponding $v_\Delta$ phasor, from $dv_\Delta/dt=-i_f/C$, is then used to initialize a small capacitor-voltage imbalance.

In [ ]:
# ============================================================
# State vector and initial conditions (Near steady state via fundamental-frequency phasor analysis)
# ============================================================
# x[0] = i_f   (filter/AC current, A)
# x[1] = v_C1  (top capacitor voltage, V)
# x[2] = v_C2  (bottom capacitor voltage, V)

w1             = 2*np.pi*f1

# m(t) = M sin(w1 t + phi) = M cos(w1 t + phi - pi/2), matching the cos reference for vg
V_ao_phasor    = (Vdc_nominal*M/2) * np.exp(1j*(phi - np.pi/2))
V_g_phasor     = np.sqrt(2)*Vg_rms * np.exp(1j*0)

I_f_phasor     = (V_ao_phasor - V_g_phasor) / (R + 1j*w1*L)
V_delta_phasor = 1j*I_f_phasor/(w1*C1)

i_f0           = np.real(I_f_phasor)
v_delta0       = np.real(V_delta_phasor)

v_C1_0         = Vdc_nominal/2 + v_delta0/2
v_C2_0         = Vdc_nominal/2 - v_delta0/2

x0             = np.array([i_f0, v_C1_0, v_C2_0])

print(f"Initializing near steady state:")
print(f"  i_f(0)  = {x0[0]:.3f} A")
print(f"  v_C1(0) = {x0[1]:.3f} V")
print(f"  v_C2(0) = {x0[2]:.3f} V")

### Switching Function and Grid Voltage

The modulation signal, triangular carrier, and sawtooth carrier are defined using `m_signal(t)`, `carrier_triangle(t)`, and `carrier_sawtooth(t)`, respectively.

In `q_signal(t)`, choose the PWM method by commenting/uncommenting the carrier line (sine-triangle PWM or sine-sawtooth PWM).

The grid voltage is defined by `vg_signal(t)`, with grid phase set to 0.

In [ ]:
def carrier_triangle(t):
    """Symmetric triangular carrier, amplitude [-1, +1], period 1/fsw."""
    Tsw = 1.0/fsw
    x = (t % Tsw)/Tsw
    return np.where(x < 0.5, 4*x - 1.0, 3.0 - 4*x)

def carrier_sawtooth(t):
    """Sawtooth carrier, amplitude [-1, +1], period 1/fsw."""
    Tsw = 1.0/fsw
    x = (t % Tsw)/Tsw
    return 2*x - 1.0

def m_signal(t):
    """Modulating reference."""
    return M*np.sin(2*np.pi*f1*t + phi)

def q_signal(t):
    """Switching function q(t) in {+1, -1}."""
    # Choose carrier by comment/uncomment:
    # carrier = carrier_triangle(t)
    carrier = carrier_sawtooth(t)
    return np.where(m_signal(t) >= carrier, 1.0, -1.0)

def vg_signal(t):
    """Grid voltage."""
    return np.sqrt(2)*Vg_rms*np.cos(2*np.pi*f1*t)

### Visualization of the Switching Function

The code below plots the modulation signal, carrier signal, and switching function over one fundamental cycle and over four switching cycles.

In [ ]:
# ============================================================
# Visualize modulation, carrier, and switching function
# ============================================================

T1  = 1.0/f1
Tsw = 1.0/fsw

# Last 1 fundamental cycle
t_fund = np.linspace(t_end - T1, t_end, 12000)
m_fund = m_signal(t_fund)
# Choose the same carrier used inside q_signal(t):
# c_fund = carrier_triangle(t_fund)
c_fund = carrier_sawtooth(t_fund)
q_fund = q_signal(t_fund)

# Last 4 switching cycles
t_zoom = np.linspace(t_end - 4*Tsw, t_end, 3000)
m_zoom = m_signal(t_zoom)
# c_zoom = carrier_triangle(t_zoom)
c_zoom = carrier_sawtooth(t_zoom)
q_zoom = q_signal(t_zoom)

fig, axs = plt.subplots(2, 2, figsize=(11, 5), sharex='col', sharey='row')

# Top row: modulation and carrier
axs[0, 0].plot(t_fund*1000, m_fund, color='red', lw=1.0, label='m(t)')
axs[0, 0].plot(t_fund*1000, c_fund, color='blue', lw=1.0, label='c(t)')
axs[0, 0].set_title('Last fundamental cycle')
axs[0, 0].set_ylabel('Amplitude')
axs[0, 0].set_ylim([-1.2, 1.2])
axs[0, 0].legend(loc='upper right')

axs[0, 1].plot(t_zoom*1000, m_zoom, color='red', lw=1.0)
axs[0, 1].plot(t_zoom*1000, c_zoom, color='blue', lw=1.0)
axs[0, 1].set_title('Last 4 switching cycles')
axs[0, 1].set_ylim([-1.2, 1.2])

# Bottom row: switching function
axs[1, 0].step(t_fund*1000, q_fund, where='post', color='black', lw=1.0, label='q(t)')
axs[1, 0].set_ylabel('q(t)')
axs[1, 0].set_xlabel('time (ms)')
axs[1, 0].set_yticks([-1, 1])
axs[1, 0].set_ylim([-1.2, 1.2])
axs[1, 0].legend(loc='upper right')

axs[1, 1].step(t_zoom*1000, q_zoom, where='post', color='black', lw=1.0)
axs[1, 1].set_xlabel('time (ms)')
axs[1, 1].set_yticks([-1, 1])
axs[1, 1].set_ylim([-1.2, 1.2])

plt.tight_layout()
plt.show()

### DC-Side Current Source

In practice, the DC-link current contributed by the upstream source-side converter contains a DC component and ripple/harmonic components. In this notebook, the ripple component is neglected and only the DC component is modeled for simplicity:

$$i_s(t)=I_{DC}.$$

The DC component is chosen from a fundamental-frequency power-balance approximation. With

$$P_{vao}=\frac{1}{2}\operatorname{Re}\!\left(V_{ao,\mathrm{phasor}}\,I_{f,\mathrm{phasor}}^{*}\right),$$

the estimate is

$$I_{DC}=\frac{P_{vao}}{V_{dc,nominal}}.$$

This choice approximately enforces average DC-link charge balance over the simulated interval. However, because this is an open-loop estimate and component losses/parasitics are not perfectly compensated, exact current balance is not guaranteed. Consequently, the average DC-link voltage may still show a slow increase or decrease.

**Important Point:** The implementation `I_DC = 1.06 * P_vao / Vdc_nominal` is one such practical estimate, where the factor `1.06` is used as a compensation margin for losses so that the average DC components are better matched.

In practical converters, closed-loop DC-link voltage control adjusts source-side power/current so that the average source current and converter DC demand are matched, thereby regulating the average DC-link voltage.

In [ ]:
# ============================================================
# DC-side current source (DC component only in this study)
# ============================================================

# Estimate I_DC from fundamental-frequency power balance
P_vao        = 0.5*np.real(V_ao_phasor*np.conj(I_f_phasor))
I_DC         = 1.06 * P_vao / Vdc_nominal

# Ripple is neglected in this model
I_ripple     = 0.0
theta_ripple = 0.0

def i_s_func(t):
    """Prescribed DC-side source current (DC component only)."""
    return I_DC + I_ripple*np.sin(2*np.pi*fsw*t + theta_ripple)

print(f"P_vao (avg. power drawn by inverter from DC side) = {P_vao:.2f} W")
print(f"I_DC (DC component of current source)             = {I_DC:.3f} A")
print(f"I_ripple (switching-frequency ripple amplitude)   = {I_ripple:.3f} A")

### Right-Hand-Side Function

The RHS function evaluates $q(t)$, grid voltage $v_g(t)$, and source current $i_s(t)$ at the current time instant. In this study, $i_s(t)$ is modeled as DC-only (ripple neglected). It then reconstructs $v_{ao}(t)$ from capacitor voltages and returns $\dot{x}$ from the governing equations:

$$v_{ao}(t)=\frac{v_{C1}-v_{C2}}{2}+q(t)\,\frac{v_{C1}+v_{C2}}{2},$$

$$\frac{di_f}{dt}=\frac{1}{L}v_{ao}(t)-\frac{R}{L}i_f-\frac{1}{L}v_g(t),$$

$$\frac{dv_{C1}}{dt}=\frac{1}{C_1}\left[i_s(t)-\frac{1+q(t)}{2}i_f\right],\qquad
\frac{dv_{C2}}{dt}=\frac{1}{C_2}\left[\frac{1-q(t)}{2}i_f+i_s(t)\right].$$

In [ ]:
# ============================================================
# Right-hand side: dx/dt = f(t, x)
# ============================================================

def rhs(t, x):
    i_f, v_C1, v_C2 = x

    q  = q_signal(t)
    vg = vg_signal(t)
    isrc = i_s_func(t)  # DC-only source current in this study

    v_ao = (v_C1 - v_C2)/2 + q*(v_C1 + v_C2)/2

    di_f = v_ao/L - (R/L)*i_f - vg/L
    dv_C1 = (isrc - (1+q)/2*i_f)/C1
    dv_C2 = ((1-q)/2*i_f + isrc)/C2

    return np.array([di_f, dv_C1, dv_C2])

### Solving the System of Differential Equations

The model is solved numerically by integrating the state-space system

$$\dot{x}(t)=f\big(t,x(t)\big),\qquad x(t)=\begin{bmatrix} i_f(t) \\ v_{C1}(t) \\ v_{C2}(t) \end{bmatrix}.$$

The solution vector returned by the solver therefore contains three trajectories:

- $i_f(t)$: injected/filter current
- $v_{C1}(t)$: upper DC-link capacitor voltage
- $v_{C2}(t)$: lower DC-link capacitor voltage

These trajectories are then used to compute derived quantities and generate the plots in the following sections.

In [ ]:
# ============================================================
# Solve the ODE
# ============================================================

max_step = t_step          # cap internal step size at our switching-resolution step

sol = solve_ivp(rhs, (t[0], t[-1]), x0, max_step=max_step, dense_output=True)

x_t = sol.sol(t)           # evaluate dense output on our time vector

i_f  = x_t[0]
v_c1 = x_t[1]
v_c2 = x_t[2]

### Deriving Other Quantities From the State Solution

The solver gives us the three states directly: $i_f(t)$, $v_{c1}(t)$, $v_{c2}(t)$. Everything else of interest is computed from these.

The total DC link voltage and the imbalance:

$$v_{dc}(t) = v_{c1}(t)+v_{c2}(t), \qquad v_\Delta(t) = v_{c1}(t)-v_{c2}(t)$$

Unlike Simulation Study 1, $v_{dc}(t)$ is **not constant** here — it is expected to show ripple (and possibly slow drift) driven by the source/load current balance, exactly as in the classical DC-link sizing analysis.

The source current $i_s(t)$ is simply the prescribed function we defined above (no longer an algebraic output of $q(t)$ and $i_f(t)$, since it is now an independent input):

The pole voltage is reconstructed from the switching function and the two capacitor voltages:

$$v_{ao}(t) = \frac{v_{c1}(t)-v_{c2}(t)}{2} + q(t)\cdot\frac{v_{c1}(t)+v_{c2}(t)}{2}$$

Finally, the instantaneous power injected into the grid is the product of grid voltage and AC current:

$$p_g(t) = v_g(t)\cdot i_f(t)$$

whose average over a fundamental cycle gives the active power delivered.

In [ ]:
# ============================================================
# Derive other quantities from the state solution
# ============================================================

q_t     = q_signal(t)
vg_t    = vg_signal(t)

v_dc    = v_c1 + v_c2

i_s     = i_s_func(t)

v_ao    = (v_c1 - v_c2)/2 + q_t*(v_c1 + v_c2)/2

p_g     = vg_t*i_f

# Common time-window masks reused by all subsequent plots
mask_1fund = (t >= (t_end - 1/f1))
mask_2fund = (t >= (t_end - 2/f1))
mask_4sw   = (t >= (t_end - 4/fsw))

### Plotting the Injected Current and Pole Voltage

To inspect the injected current $i_f(t)$ and the pole voltage $v_{ao}(t)$ at different timescales in one view, we show a 2x3 grid: the first row contains $i_f(t)$ and the second row contains $v_{ao}(t)$. Each row is shown for the full simulated waveform, the last two fundamental cycles, and the last four switching cycles.

In [ ]:
# ============================================================
# Plot injected current and pole voltage at three timescales
# ============================================================

fig, axs = plt.subplots(2, 3, figsize=(16, 7))

# Row 1: injected current i_f(t)
axs[0, 0].plot(t*1000, i_f, color='blue', lw=1.0)
axs[0, 0].set_title('Complete simulation')
axs[0, 0].set_ylabel('$i_f$ (A)')

axs[0, 1].plot(t[mask_2fund]*1000, i_f[mask_2fund], color='blue', lw=1.0)
axs[0, 1].set_title('Last 2 fundamental cycles')
axs[0, 1].set_ylabel('$i_f$ (A)')

axs[0, 2].plot(t[mask_4sw]*1000, i_f[mask_4sw], color='blue', lw=1.0)
axs[0, 2].set_title('Last 4 switching cycles')
axs[0, 2].set_ylabel('$i_f$ (A)')

# Row 2: pole voltage v_ao(t)
axs[1, 0].plot(t*1000, v_ao, color='red', lw=1.0)
axs[1, 0].set_ylabel('$v_{ao}$ (V)')
axs[1, 0].set_xlabel('time (ms)')

axs[1, 1].plot(t[mask_2fund]*1000, v_ao[mask_2fund], color='red', lw=1.0)
axs[1, 1].set_ylabel('$v_{ao}$ (V)')
axs[1, 1].set_xlabel('time (ms)')

axs[1, 2].plot(t[mask_4sw]*1000, v_ao[mask_4sw], color='red', lw=1.0)
axs[1, 2].set_ylabel('$v_{ao}$ (V)')
axs[1, 2].set_xlabel('time (ms)')

plt.tight_layout()
plt.show()

### Plotting Capacitor Voltages and Total DC-Link Voltage

The capacitor voltages $v_{c1}(t)$ and $v_{c2}(t)$ are plotted together with the total DC-link voltage $v_{dc}(t)=v_{c1}(t)+v_{c2}(t)$. This shows the total instantaneous voltage across the DC link.

In [ ]:
# ============================================================
# Plot capacitor voltages and total DC-link voltage at three timescales
# ============================================================

fig, axs = plt.subplots(3, 3, figsize=(16, 10))

# Row 1: v_c1(t)
axs[0, 0].plot(t*1000, v_c1, color='red', lw=1.0)
axs[0, 0].set_title('Complete simulation')
axs[0, 0].set_ylabel('$v_{c1}$ (V)')

axs[0, 1].plot(t[mask_2fund]*1000, v_c1[mask_2fund], color='red', lw=1.0)
axs[0, 1].set_title('Last 2 fundamental cycles')
axs[0, 1].set_ylabel('$v_{c1}$ (V)')

axs[0, 2].plot(t[mask_4sw]*1000, v_c1[mask_4sw], color='red', lw=1.0)
axs[0, 2].set_title('Last 4 switching cycles')
axs[0, 2].set_ylabel('$v_{c1}$ (V)')

# Row 2: v_c2(t)
axs[1, 0].plot(t*1000, v_c2, color='green', lw=1.0)
axs[1, 0].set_ylabel('$v_{c2}$ (V)')

axs[1, 1].plot(t[mask_2fund]*1000, v_c2[mask_2fund], color='green', lw=1.0)
axs[1, 1].set_ylabel('$v_{c2}$ (V)')

axs[1, 2].plot(t[mask_4sw]*1000, v_c2[mask_4sw], color='green', lw=1.0)
axs[1, 2].set_ylabel('$v_{c2}$ (V)')

# Row 3: v_dc(t)
axs[2, 0].plot(t*1000, v_dc, color='blue', lw=1.0)
axs[2, 0].set_ylabel('$v_{dc}$ (V)')
axs[2, 0].set_xlabel('time (ms)')

axs[2, 1].plot(t[mask_2fund]*1000, v_dc[mask_2fund], color='blue', lw=1.0)
axs[2, 1].set_ylabel('$v_{dc}$ (V)')
axs[2, 1].set_xlabel('time (ms)')

axs[2, 2].plot(t[mask_4sw]*1000, v_dc[mask_4sw], color='blue', lw=1.0)
axs[2, 2].set_ylabel('$v_{dc}$ (V)')
axs[2, 2].set_xlabel('time (ms)')

plt.tight_layout()
plt.show()

### Plotting Source and Rail Currents

The prescribed source current $i_s(t)$ and rail currents

$$i_t(t)=\frac{1+q(t)}{2}i_f(t),\qquad i_b(t)=\frac{1-q(t)}{2}i_f(t),$$

are plotted at the same three time scales used previously.

In [ ]:
# ============================================================
# Compute and plot source current and rail currents
# ============================================================

i_t = (1 + q_t)/2 * i_f
i_b = (1 - q_t)/2 * i_f

fig, axs = plt.subplots(3, 3, figsize=(16, 11))

# Row 1: source current i_f(t)
axs[0, 0].plot(t*1000, i_f, color='brown', lw=1.0)
axs[0, 0].set_title('Complete simulation')
axs[0, 0].set_ylabel('$i_f$ (A)')

axs[0, 1].plot(t[mask_2fund]*1000, i_f[mask_2fund], color='brown', lw=1.0)
axs[0, 1].set_title('Last 2 fundamental cycles')
axs[0, 1].set_ylabel('$i_f$ (A)')

axs[0, 2].plot(t[mask_4sw]*1000, i_f[mask_4sw], color='brown', lw=1.0)
axs[0, 2].set_title('Last 4 switching cycles')
axs[0, 2].set_ylabel('$i_f$ (A)')

# Row 2: top-rail current i_t(t)
axs[1, 0].plot(t*1000, i_t, color='red', lw=1.0)
axs[1, 0].set_ylabel('$i_t$ (A)')

axs[1, 1].plot(t[mask_2fund]*1000, i_t[mask_2fund], color='red', lw=1.0)
axs[1, 1].set_ylabel('$i_t$ (A)')

axs[1, 2].plot(t[mask_4sw]*1000, i_t[mask_4sw], color='red', lw=1.0)
axs[1, 2].set_ylabel('$i_t$ (A)')

# Row 3: bottom-rail current i_b(t)
axs[2, 0].plot(t*1000, i_b, color='green', lw=1.0)
axs[2, 0].set_ylabel('$i_b$ (A)')
axs[2, 0].set_xlabel('time (ms)')

axs[2, 1].plot(t[mask_2fund]*1000, i_b[mask_2fund], color='green', lw=1.0)
axs[2, 1].set_ylabel('$i_b$ (A)')
axs[2, 1].set_xlabel('time (ms)')

axs[2, 2].plot(t[mask_4sw]*1000, i_b[mask_4sw], color='green', lw=1.0)
axs[2, 2].set_ylabel('$i_b$ (A)')
axs[2, 2].set_xlabel('time (ms)')

plt.tight_layout()
plt.show()

### Plotting the Instantaneous Power

The instantaneous grid power

$$p_g(t)=v_g(t)i_f(t),$$

is plotted at the same three time scales. The average active power over the final fundamental cycle is also computed:

$$P_{avg}=\langle p_g(t)\rangle_{T_1}.$$

In [ ]:
# ============================================================
# Plot instantaneous power
# ============================================================

fig, axs = plt.subplots(1, 3, figsize=(16, 4.5))

# Full simulation
axs[0].plot(t*1000, p_g, color='orange', lw=1.0)
axs[0].set_xlabel('time (ms)')
axs[0].set_ylabel('$p_g$ (W)')
axs[0].set_title('Complete simulation')

# Last 2 fundamental cycles
axs[1].plot(t[mask_2fund]*1000, p_g[mask_2fund], color='orange', lw=1.0)
axs[1].set_xlabel('time (ms)')
axs[1].set_title('Last 2 fundamental cycles')

# Last 4 switching cycles
axs[2].plot(t[mask_4sw]*1000, p_g[mask_4sw], color='orange', lw=1.0)
axs[2].set_xlabel('time (ms)')
axs[2].set_title('Last 4 switching cycles')

plt.tight_layout()
plt.show()

# Average active power over the last fundamental cycle
P_avg = np.mean(p_g[mask_1fund])
print(f"Average active power over last fundamental cycle: P_avg = {P_avg:.2f} W")

### Credits and Disclaimer

This notebook is part of the NPTEL course *Grid Connected Power Converters - Operating Principles*.

The simulation ideas, modelling approach, and technical interpretation in this notebook are original to the author.

This notebook presents a simplified current-source-driven switching-pole converter model. It assumes a prescribed DC-side source current, neglects source-current ripple and other higher-order effects, and uses approximate initialization and power-balance estimates that are appropriate for pedagogical study of waveform behavior. The results are therefore useful for understanding the operating principle and analysis method, but they are not a substitute for detailed converter design validation.

AI assistance was used only to polish and refine portions of the code structure and documentation language.